In [22]:
!pip -q install shap

In [23]:
import warnings; warnings.filterwarnings("ignore")
from itertools import product
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier, LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.semi_supervised import LabelSpreading
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_squared_error,
                             silhouette_score, silhouette_samples)
import shap

SEED = 42
np.random.seed(SEED)

# VERİ

## Preprocessing

In [24]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
features = list(X.columns)
FP_FEATURE = "mean radius"
X.shape

(569, 30)

## Split

In [25]:
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_t2, X_val, y_t2, y_val = train_test_split(X_tmp, y_tmp, test_size=0.20, stratify=y_tmp, random_state=SEED)
X_train, X_traindev, y_train, y_traindev = train_test_split(X_t2, y_t2, test_size=0.20, stratify=y_t2, random_state=SEED)

scaler = StandardScaler().fit(X_train)
X_train, X_traindev, X_val, X_test = map(scaler.transform, (X_train, X_traindev, X_val, X_test))
y_train, y_traindev, y_val, y_test = (s.to_numpy() for s in (y_train, y_traindev, y_val, y_test))
X_train.shape, X_traindev.shape, X_val.shape, X_test.shape

((291, 30), (73, 30), (91, 30), (114, 30))

# MODEL

In [26]:
fitted = {}

def tune(estimator, grid, Xtr, ytr, Xval, yval):
    keys = list(grid); best_s, best_p = -np.inf, None
    for combo in product(*[grid[k] for k in keys]):
        p = dict(zip(keys, combo))
        s = accuracy_score(yval, clone(estimator).set_params(**p).fit(Xtr, ytr).predict(Xval))
        if s > best_s: best_s, best_p = s, p
    return clone(estimator).set_params(**best_p).fit(Xtr, ytr), best_p

## Linear Models

In [27]:
# Training
model = RidgeClassifier().fit(X_train, y_train)
# Tuning
fitted["Linear Models"], best = tune(RidgeClassifier(),
    {"alpha": [0.01, 0.1, 1, 10, 100]}, X_train, y_train, X_val, y_val); best

{'alpha': 10}

## Logistic Regression

In [28]:
# Training
model = LogisticRegression(max_iter=5000).fit(X_train, y_train)
# Tuning
fitted["Logistic Regression"], best = tune(LogisticRegression(max_iter=5000),
    {"C": [0.01, 0.1, 1, 10, 100]}, X_train, y_train, X_val, y_val); best

{'C': 0.1}

## Support Vector Machine

In [29]:
# Training
model = SVC(kernel="rbf").fit(X_train, y_train)
# Tuning
fitted["SVM"], best = tune(SVC(kernel="rbf"),
    {"C": [0.1, 1, 10, 100], "gamma": [0.001, 0.01, 0.1, "scale"]}, X_train, y_train, X_val, y_val); best

{'C': 1, 'gamma': 0.1}

## k-NN

In [30]:
# Training
model = KNeighborsClassifier().fit(X_train, y_train)
# Tuning
fitted["k-NN"], best = tune(KNeighborsClassifier(),
    {"n_neighbors": [3, 5, 7, 9, 11, 15]}, X_train, y_train, X_val, y_val); best

{'n_neighbors': 3}

## Decision Tree

In [31]:
# Training
model = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)
# Tuning
fitted["Decision Tree"], best = tune(DecisionTreeClassifier(random_state=SEED),
    {"max_depth": [2, 3, 4, 5, 8, None]}, X_train, y_train, X_val, y_val); best

{'max_depth': 3}

## Random Forest

In [32]:
# Training
model = RandomForestClassifier(random_state=SEED, n_jobs=-1).fit(X_train, y_train)
# Tuning
fitted["Random Forest"], best = tune(RandomForestClassifier(random_state=SEED, n_jobs=-1),
    {"n_estimators": [200, 400], "max_depth": [None, 6, 10]}, X_train, y_train, X_val, y_val); best

{'n_estimators': 200, 'max_depth': None}

## Gradient Boosting

In [33]:
# Training
model = GradientBoostingClassifier(random_state=SEED).fit(X_train, y_train)
# Tuning
fitted["Gradient Boosting"], best = tune(GradientBoostingClassifier(random_state=SEED),
    {"learning_rate": [0.05, 0.1], "n_estimators": [150, 300], "subsample": [1.0, 0.8]},
    X_train, y_train, X_val, y_val); best

{'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}

## Label Propagation

In [34]:
rng = np.random.RandomState(SEED)
y_semi = y_train.copy(); y_semi[rng.rand(len(y_semi)) < 0.5] = -1
# Training
model = LabelSpreading(kernel="rbf", gamma=20, max_iter=1000).fit(X_train, y_semi)
# Tuning
best_s = -np.inf
for g in [1, 5, 10, 20, 50]:
    m = LabelSpreading(kernel="rbf", gamma=g, max_iter=1000).fit(X_train, y_semi)
    s = accuracy_score(y_val, m.predict(X_val))
    if s > best_s: best_s, fitted["Label Propagation"], best = s, m, {"gamma": g}
best

{'gamma': 1}

## K-Means

In [35]:
# Training
model = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(X_train)
# Tuning
best_s = -np.inf
for k in [2, 3, 4, 5, 6]:
    m = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_train)
    s = silhouette_score(X_train, m.labels_)
    if s > best_s: best_s, kmeans, best = s, m, {"k": k, "J(k)": round(m.inertia_, 2)}
best

{'k': 2, 'J(k)': 5690.53}

## Feature Prediction

In [36]:
j = features.index(FP_FEATURE)
Xtr_fp, ytr_fp   = np.delete(X_train, j, 1), X_train[:, j]
Xval_fp, yval_fp = np.delete(X_val, j, 1),   X_val[:, j]
Xte_fp,  yte_fp  = np.delete(X_test, j, 1),  X_test[:, j]
# Training
model = Ridge(alpha=1.0).fit(Xtr_fp, ytr_fp)
# Tuning
best_mse = np.inf
for a in [0.01, 0.1, 1, 10, 100]:
    m = Ridge(alpha=a).fit(Xtr_fp, ytr_fp)
    mse = mean_squared_error(yval_fp, m.predict(Xval_fp))
    if mse < best_mse: best_mse, fp, best = mse, m, {"alpha": a}
best

{'alpha': 0.01}

## Q-Learning

In [37]:
corr = np.array([np.corrcoef(X_train[:, c], y_train)[0, 1] for c in range(X_train.shape[1])])
cols, NB = np.argsort(-np.abs(corr))[:2], 5
edges = [np.quantile(X_train[:, c], np.linspace(0, 1, NB + 1)) for c in cols]
NS = NB ** len(cols)
def to_states(Xmat):
    s = np.zeros(len(Xmat), dtype=int)
    for c, e in zip(cols, edges):
        s = s * NB + np.clip(np.digitize(Xmat[:, c], e[1:-1]), 0, NB - 1)
    return s
s_train, s_val, s_test = map(to_states, (X_train, X_val, X_test))

def q_learn(s, ylab, alpha, gamma, eps, episodes=200, seed=SEED):
    r = np.random.RandomState(seed); Q = np.zeros((NS, 2)); idx = np.arange(len(s))
    for _ in range(episodes):
        r.shuffle(idx)
        for t in range(len(idx)):
            st = s[idx[t]]
            a  = r.randint(2) if r.rand() < eps else int(np.argmax(Q[st]))
            rew = 1.0 if a == ylab[idx[t]] else -1.0
            sn = s[idx[(t + 1) % len(idx)]]
            Q[st, a] += alpha * (rew + gamma * np.max(Q[sn]) - Q[st, a])
    return Q
q_predict = lambda Q, s: np.argmax(Q[s], axis=1)

# Training
Q = q_learn(s_train, y_train, alpha=0.1, gamma=0.9, eps=0.1)
# Tuning
best_s = -np.inf
for al, ga, ep in product([0.1, 0.3], [0.0, 0.9], [0.1, 0.2]):
    Qc = q_learn(s_train, y_train, al, ga, ep)
    s = accuracy_score(y_val, q_predict(Qc, s_val))
    if s > best_s: best_s, Qbest, best = s, Qc, {"alpha": al, "gamma": ga, "eps": ep}
q_test_pred = q_predict(Qbest, s_test); best

{'alpha': 0.1, 'gamma': 0.0, 'eps': 0.1}

# DEĞERLENDİRME

## Errors

In [38]:
splits = {"train": (X_train, y_train), "train-dev": (X_traindev, y_traindev),
          "val": (X_val, y_val), "test": (X_test, y_test)}
fp_splits = {"train": (Xtr_fp, ytr_fp),
             "train-dev": (np.delete(X_traindev, j, 1), X_traindev[:, j]),
             "val": (Xval_fp, yval_fp), "test": (Xte_fp, yte_fp)}
km_splits = {"train": X_train, "train-dev": X_traindev, "val": X_val, "test": X_test}
q_splits  = {"train": (s_train, y_train), "train-dev": (to_states(X_traindev), y_traindev),
             "val": (s_val, y_val), "test": (s_test, y_test)}

diag = {}
for n, mdl in fitted.items():
    diag[n] = {"metric": "error", **{s: round(1 - accuracy_score(yy, mdl.predict(XX)), 4) for s, (XX, yy) in splits.items()}}
diag["Q-Learning"] = {"metric": "error", **{s: round(1 - accuracy_score(yy, q_predict(Qbest, ss)), 4) for s, (ss, yy) in q_splits.items()}}
diag["Feature Prediction"] = {"metric": "MSE", **{s: round(mean_squared_error(yy, fp.predict(XX)), 4) for s, (XX, yy) in fp_splits.items()}}
diag["K-Means"] = {"metric": "silhouette", **{s: round(silhouette_score(XX, kmeans.predict(XX)), 4) for s, XX in km_splits.items()}}
pd.DataFrame(diag).T[["metric", "train", "train-dev", "val", "test"]]

,metric,train,train-dev,val,test
Linear Models,error,0.0412,0.0274,0.0549,0.0263
Logistic Regression,error,0.0206,0.0274,0.033,0.0263
SVM,error,0.0137,0.0274,0.011,0.0351
k-NN,error,0.0206,0.0548,0.022,0.0439
Decision Tree,error,0.0206,0.0959,0.044,0.0614
Random Forest,error,0.0,0.0548,0.044,0.0526
Gradient Boosting,error,0.0,0.0685,0.033,0.0526
Label Propagation,error,0.0275,0.0411,0.044,0.0439
Q-Learning,error,0.0653,0.1507,0.0769,0.0439
Feature Prediction,MSE,0.0002,0.0011,0.0003,0.0006


## Classification Prediction

In [39]:
def clf_metrics(yt, yp):
    return {"Accuracy": accuracy_score(yt, yp), "Precision": precision_score(yt, yp),
            "Recall": recall_score(yt, yp), "F1": f1_score(yt, yp)}
preds = {n: mdl.predict(X_test) for n, mdl in fitted.items()}
preds["Q-Learning"] = q_test_pred
clf_df = pd.DataFrame({n: clf_metrics(y_test, p) for n, p in preds.items()}).T.sort_values("F1", ascending=False).round(4)
clf_df

,Accuracy,Precision,Recall,F1
Linear Models,0.9737,0.9600,1.0000,0.9796
Logistic Regression,0.9737,0.9726,0.9861,0.9793
SVM,0.9649,0.9857,0.9583,0.9718
Label Propagation,0.9561,0.9589,0.9722,0.9655
k-NN,0.9561,0.9718,0.9583,0.9650
Q-Learning,0.9561,0.9718,0.9583,0.9650
Gradient Boosting,0.9474,0.9459,0.9722,0.9589
Random Forest,0.9474,0.9583,0.9583,0.9583
Decision Tree,0.9386,0.9577,0.9444,0.9510


In [40]:
best = clf_df.index[0]
pd.DataFrame(confusion_matrix(y_test, preds[best]),
             index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"])

,pred 0,pred 1
actual 0,39,3
actual 1,0,72


## Regression Prediction

In [41]:
yhat = fp.predict(Xte_fp); e = yte_fp - yhat
MAE  = np.mean(np.abs(e)); MSE = np.mean(e ** 2); SSE = np.sum(e ** 2)
RMSE = np.sqrt(MSE); SST = np.sum((yte_fp - yte_fp.mean()) ** 2); R2 = 1 - SSE / SST
pd.Series({"MAE": MAE, "MSE": MSE, "RMSE": RMSE, "SSE": SSE, "SST": SST, "R2": R2}).round(4)

,0
MAE,0.0154
MSE,0.0006
RMSE,0.0254
SSE,0.0734
SST,123.5235
R2,0.9994


## Clustering Prediction

In [42]:
labels = kmeans.predict(X_test)
S = silhouette_score(X_test, labels); Si = silhouette_samples(X_test, labels)
tag = "iyi ayrılmış (S≈1)" if S > 0.5 else ("sınırda (S≈0)" if S >= 0 else "yanlış kümelenmiş (S<0)")
pd.Series({"k": kmeans.n_clusters, "Silhouette": round(S, 4), "yorum": tag})

,0
k,2
Silhouette,0.3263
yorum,sınırda (S≈0)


# AÇIKLANABİLİRLİK

In [43]:
bg = shap.kmeans(X_train, 20);  Xe = X_test[:40]
bg_fp = shap.kmeans(Xtr_fp, 20);  Xe_fp = Xte_fp[:40]
feat_fp = [f for f in features if f != FP_FEATURE]

def km_soft(A):
    d = kmeans.transform(np.asarray(A)); z = -d; z -= z.max(1, keepdims=True)
    p = np.exp(z); p /= p.sum(1, keepdims=True); return p[:, 1]
def q_margin(A):
    s = to_states(np.asarray(A)); return Qbest[s, 1] - Qbest[s, 0]

score = {
 "Linear Models":       (fitted["Linear Models"].decision_function,               bg, Xe, features),
 "Logistic Regression": (fitted["Logistic Regression"].decision_function,         bg, Xe, features),
 "SVM":                 (fitted["SVM"].decision_function,                         bg, Xe, features),
 "k-NN":                (lambda A: fitted["k-NN"].predict_proba(A)[:, 1],         bg, Xe, features),
 "Decision Tree":       (lambda A: fitted["Decision Tree"].predict_proba(A)[:, 1], bg, Xe, features),
 "Random Forest":       (lambda A: fitted["Random Forest"].predict_proba(A)[:, 1], bg, Xe, features),
 "Gradient Boosting":   (fitted["Gradient Boosting"].decision_function,           bg, Xe, features),
 "Label Propagation":   (lambda A: fitted["Label Propagation"].predict_proba(A)[:, 1], bg, Xe, features),
 "K-Means":             (km_soft,   bg,    Xe,    features),
 "Feature Prediction":  (fp.predict, bg_fp, Xe_fp, feat_fp),
 "Q-Learning":          (q_margin,  bg,    Xe,    features),
}

In [44]:
shap_imp, recon_ok = {}, {}
for name, (fn, back, Xexp, names) in score.items():
    ke = shap.KernelExplainer(fn, back)
    sv = np.array(ke.shap_values(Xexp, nsamples=100, silent=True))
    if sv.ndim == 3: sv = sv[..., 1] if sv.shape[-1] > 1 else sv[..., 0]
    phi0 = float(np.array(ke.expected_value).ravel()[0])
    recon_ok[name] = bool(np.allclose(phi0 + sv.sum(1), fn(Xexp), atol=1e-2))
    shap_imp[name] = pd.Series(np.abs(sv).mean(0), index=names).sort_values(ascending=False)
pd.Series(recon_ok, name="f(x)=φ0+Σφi")

,f(x)=φ0+Σφi
Linear Models,True
Logistic Regression,True
SVM,True
k-NN,True
Decision Tree,True
Random Forest,True
Gradient Boosting,True
Label Propagation,True
K-Means,True
Feature Prediction,True


In [45]:
pd.DataFrame({n: s.head(5).index.tolist() for n, s in shap_imp.items()},
             index=[f"φ rank {i + 1}" for i in range(5)]).T

,φ rank 1,φ rank 2,φ rank 3,φ rank 4,φ rank 5
Linear Models,worst concave points,radius error,worst fractal dimension,worst radius,mean concave points
Logistic Regression,worst texture,worst perimeter,worst radius,mean concave points,worst area
SVM,worst texture,worst perimeter,worst radius,worst smoothness,worst area
k-NN,worst radius,worst texture,mean concavity,worst perimeter,mean perimeter
Decision Tree,worst area,worst perimeter,worst concave points,mean concavity,worst texture
Random Forest,worst perimeter,worst area,worst concave points,worst radius,mean concave points
Gradient Boosting,worst perimeter,worst concave points,worst area,mean concave points,worst texture
Label Propagation,worst radius,worst texture,worst area,worst concave points,area error
K-Means,worst concave points,mean concavity,worst radius,worst concavity,worst perimeter
Feature Prediction,mean perimeter,worst radius,worst area,worst perimeter,mean area
